# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
before = len(df)
print('exact duplicate rows:', df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

removed = before - len(df)
log('dedupe', 'dropped exact duplicate rows (all columns match); reset index', removed)

exact duplicate rows: 15
[dedupe] dropped exact duplicate rows (all columns match); reset index (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
print(df['price'].value_counts())

had_dollar = df['price'].str.startswith('$').sum()

df['price'] = pd.to_numeric(df['price'].str.replace('$', '', regex=False).str.strip(),
                            errors='coerce')

assert df['price'].isna().sum() == 0, 'some prices failed to parse'
print(df['price'].value_counts())

log('price', "stripped '$' and converted price to float", int(had_dollar))

price
6.0       67
$12.00    65
$7.50     59
7.5       58
24        51
Name: count, dtype: int64
price
7.5     117
6.0      67
12.0     65
24.0     51
Name: count, dtype: int64
[price] stripped '$' and converted price to float (124 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

missing = df['qty'].isna().sum()
negative = (df['qty'] < 1).sum()
print('missing qty:', missing, '| qty < 1:', negative)

before = len(df)
df = df[df['qty'].notna() & (df['qty'] >= 1)].copy()
df['qty'] = df['qty'].astype(int)

log('qty-missing', 'dropped rows with missing qty', int(missing))
log('qty-negative', 'dropped rows with qty < 1 (the -1 values)', int(negative))
assert before - len(df) == missing + negative

missing qty: 12 | qty < 1: 13
[qty-missing] dropped rows with missing qty (12 row(s))
[qty-negative] dropped rows with qty < 1 (the -1 values) (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())
ITEM_MAP = {'cheese burger': 'Cheeseburger', 'foam finger': 'Foam Finger', 'rain poncho': 'Rain Poncho'}

changed = df['item'].isin(ITEM_MAP).sum()
df['item'] = df['item'].replace(ITEM_MAP)

assert set(df['item']) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
log('item', 'collapsed 6 spellings into 3 products', int(changed))

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] collapsed 6 spellings into 3 products (126 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO
print(df['category'].value_counts())
CAT_MAP = {'food': 'Food', 'rain-gear': 'RainGear', 'Apparel': 'Merch'}

apparel = (df['category'] == 'Apparel').sum()
spelling = df['category'].isin(['food', 'rain-gear']).sum()
df['category'] = df['category'].replace(CAT_MAP)

assert set(df['category']) == {'Food', 'Merch', 'RainGear'}
log('category', 'fixed case/hyphen variants (food, rain-gear)', int(spelling))
log('category', 'business call: folded Apparel into Merch', int(apparel))

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] fixed case/hyphen variants (food, rain-gear) (89 row(s))
[category] business call: folded Apparel into Merch (43 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
# TODO: assert something about item
assert set(df['item']) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
# TODO: assert something about category
assert set(df['category']) == {'Food', 'Merch', 'RainGear'}
print('clean:', df.shape)


clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO
df['revenue'] = (df['qty'] * df['price']).round(2)

by_cat = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
print(by_cat)
print(f"Total: ${df['revenue'].sum():,.2f}")

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64
Total: $4,740.00


**What I would tell the vendor:** They should keep more stock of food, as it produced the most revenue.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,dedupe,dropped exact duplicate rows (all columns matc...,15
1,price,stripped '$' and converted price to float,124
2,qty-missing,dropped rows with missing qty,12
3,qty-negative,dropped rows with qty < 1 (the -1 values),13
4,item,collapsed 6 spellings into 3 products,126
5,category,"fixed case/hyphen variants (food, rain-gear)",89
6,category,business call: folded Apparel into Merch,43


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a: Deduplication changed the total the most and dropped from 4852.50 to 4594.50.

b: I kept the labeled categories with Apparel folded into Merch. Deriving category from item instead would keep the 4740 total but flip the ranking, because RainGear would lead at 1729.50 and Food would drop from first 1656.00 to last 1417.50. I kept the labels because replacing recorded data with an inferred mapping is a bigger assumption, but the Food first result is weak.